# MCP: AI ko tools se jodne ka standard
**AI Explained · Episode 20 (full version)** · Jajati AI

Har cell ko upar se neeche chalao (Shift + Enter). Google Colab mein free chalta hai.

Is notebook mein koi asli AI nahi chalta, koi API key nahi chahiye, aur asli MCP SDK bhi nahi. Ek **toy** MCP ka shape dikhata hai: server, client, aur do messages, `tools/list` aur `tools/call`. Apps, tools aur weather ka data hamne teaching ke liye banaye hain. Asli MCP mein aur bhi hai (stdio ya HTTP transport, auth, resources, prompts), par tareeka yahi hai.

In [1]:
APPS = ["chat app", "code editor",
        "terminal agent"]
TOOLS = ["weather", "calendar",
         "github", "drive"]

# bina standard: har app, har tool ka
# apna jugaad code
glue = [(a, t) for a in APPS for t in TOOLS]
print(len(APPS), "apps,", len(TOOLS), "tools")
print("bina standard:", len(glue), "connectors")

# MCP: har tool ek server, har app ek client
mcp = len(TOOLS) + len(APPS)
print("MCP ke saath :", mcp,
      "(4 servers + 3 clients)")

3 apps, 4 tools
bina standard: 12 connectors
MCP ke saath : 7 (4 servers + 3 clients)


In [2]:
APPS.append("phone assistant")   # chautha app
glue = [(a, t) for a in APPS for t in TOOLS]
print(len(APPS), "apps,", len(TOOLS), "tools")
print("bina standard:", len(glue))
print("MCP ke saath :", len(TOOLS) + len(APPS))

4 apps, 4 tools
bina standard: 16
MCP ke saath : 8


In [3]:
WEATHER = {"pune": 31, "delhi": 24}  # nakli

def weather_server(msg):       # MCP server
    m, p = msg["method"], msg["params"]
    if m == "tools/list":
        out = {"tools": [{
          "name": "get_weather",
          "description": "city ka weather",
          "inputSchema": {"city": "string"}}]}
    if m == "tools/call":
        c = p["arguments"]["city"]
        out = {"content": f"{c}: {WEATHER[c]}°C"}
    return {"jsonrpc": "2.0", "id": msg["id"],
            "result": out}

In [4]:
def client(server, method, params={}):
    msg = {"jsonrpc": "2.0", "id": 1,
           "method": method, "params": params}
    return server(msg)["result"]   # ek hi code

tools = client(weather_server, "tools/list")
print([t["name"] for t in tools["tools"]])

['get_weather']


In [5]:
ASK = {"name": "get_weather",
       "arguments": {"city": "pune"}}
for app in ["chat app", "code editor"]:
    r = client(weather_server, "tools/call", ASK)
    print(app, "->", r["content"])

chat app -> pune: 31°C
code editor -> pune: 31°C


In [6]:
LOG = []                       # jo sach mein chala

def calendar_server(msg):      # naya server
    m, p = msg["method"], msg["params"]
    if m == "tools/list":
        out = {"tools": [{
          "name": "book_slot",
          "description": "interview slot book",
          "inputSchema": {"day": "string"}}]}
    if m == "tools/call":
        LOG.append("book_slot")
        d = p["arguments"]["day"]
        out = {"content": f"slot: {d}, 4 pm"}
    return {"jsonrpc": "2.0", "id": msg["id"],
            "result": out}

SERVERS = [weather_server, calendar_server]
MENU = {}                      # tool -> server
for s in SERVERS:              # app ka code same
    for t in client(s, "tools/list")["tools"]:
        MENU[t["name"]] = s
print(len(SERVERS), "servers, tools:", list(MENU))

2 servers, tools: ['get_weather', 'book_slot']


In [7]:
def toy_model(q):     # Ep 19 wala toy: sirf likhta
    w = q.lower().split()
    if "weather" in w:
        return {"name": "get_weather",
                "arguments": {"city": w[0]}}
    if "slot" in w:
        return {"name": "book_slot",
                "arguments": {"day": w[0]}}

call = toy_model("pune ka weather kaisa hai")
print(call)
server = MENU[call["name"]]   # app chunta hai
print(client(server, "tools/call", call))

{'name': 'get_weather', 'arguments': {'city': 'pune'}}
{'content': 'pune: 31°C'}


In [8]:
def run(call, user_ok):        # app ka gate
    if call["name"] == "book_slot" and not user_ok:
        return "user ne mana kiya, nahi chala"
    s = MENU[call["name"]]
    return client(s, "tools/call", call)["content"]

call = toy_model("friday ko slot book karo")
print(run(call, user_ok=False))
print("chala:", LOG)
print(run(call, user_ok=True))
print("chala:", LOG)

user ne mana kiya, nahi chala
chala: []
slot: friday, 4 pm
chala: ['book_slot']


### Asli duniya mein
MCP spec: *"Model Context Protocol (MCP) is an open protocol that enables seamless integration between LLM applications and external data sources and tools."* Teen roles: **host** (AI app), **client** (host ke andar connector, har server ke liye ek), **server** (tools aur data deta hai). Messages JSON-RPC 2.0 mein hote hain. Tools ke liye do methods: `tools/list` aur `tools/call`. Local server stdio par chalta hai, remote server Streamable HTTP par.

MCP ki site: *"Think of MCP like a USB-C port for AI applications."* Anthropic ne ise 25 Nov 2024 ko launch kiya: *"Every new data source requires its own custom implementation"*, aur MCP *"replacing fragmented integrations with a single protocol"*. 9 Dec 2025 ko Anthropic ne MCP Linux Foundation ke naye Agentic AI Foundation ko de diya. Linux Foundation ke mutabiq Claude, Cursor, Microsoft Copilot, Gemini, VS Code aur ChatGPT ise use karte hain.

MCP function calling ki jagah nahi leta. Model ab bhi call likhta hai (Episode 19); MCP batata hai ki app tools kaise dhoondhe aur kaise chalaye. Aur action se pehle insaan: spec kehta hai *"there SHOULD always be a human in the loop with the ability to deny tool invocations."* Sirf un servers ko jodo jin par bharosa ho: ek galat server tumhara data nikaal sakta hai.

In [9]:
ANSWER = ("MCP is an open protocol that "
  "standardises how an AI app connects to "
  "tools and data. Each tool is wrapped "
  "once as an MCP server; each app speaks "
  "MCP once as a client, so M apps and N "
  "tools need M + N pieces, not M x N. "
  "The client finds tools with tools/list "
  "and runs one with tools/call. It does "
  "not replace function calling: the "
  "model still writes the call, and a "
  "human should approve risky actions.")
print(ANSWER)

MCP is an open protocol that standardises how an AI app connects to tools and data. Each tool is wrapped once as an MCP server; each app speaks MCP once as a client, so M apps and N tools need M + N pieces, not M x N. The client finds tools with tools/list and runs one with tools/call. It does not replace function calling: the model still writes the call, and a human should approve risky actions.


In [10]:
# TUMHARI BAARI: apna server banao, app same
def canteen_server(msg):
    out = {"tools": [{"name": "canteen_menu",
      "description": "aaj ka khana",
      "inputSchema": {"day": "string"}}]}
    return {"jsonrpc": "2.0", "id": msg["id"],
            "result": out}

SERVERS.append(canteen_server)
for s in SERVERS:
    for t in client(s, "tools/list")["tools"]:
        MENU[t["name"]] = s
print(len(SERVERS), "servers, tools:", list(MENU))

3 servers, tools: ['get_weather', 'book_slot', 'canteen_menu']
